# Session 4 · Migrate state, then break it

Queen City Trip Analytics, a fictional South End, Charlotte firm, gains a second engineer and moves its state off one laptop. A bucket, a topic and a generated password on local state, moved to a versioned backend and then broken eight ways.

In [ ]:
source ~/dsba6190-live-demo-04/env.sh && cd "$WORKDIR" && echo "$PROJECT" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · Read the state you already have

In [ ]:
terraform state list

In [ ]:
terraform output

In [ ]:
terraform state show random_password.db_admin

In [ ]:
grep -o '"result": "[^"]*"' terraform.tfstate

## Step 2 · Bootstrap the state bucket, with versioning

In [ ]:
gcloud storage buckets create "gs://$STATE_BUCKET" \
  --project "$PROJECT" --location=US-EAST1 \
  --uniform-bucket-level-access --public-access-prevention

gcloud storage buckets update "gs://$STATE_BUCKET" \
  --project "$PROJECT" --versioning

In [ ]:
gcloud storage buckets describe "gs://$STATE_BUCKET" \
  --project "$PROJECT" \
  --format="yaml(name,location,default_storage_class,versioning_enabled,uniform_bucket_level_access,public_access_prevention)"

## Step 3 · The backend block, and the migration

In [ ]:
cp backend.tf.staged backend.tf
cat backend.tf

In [ ]:
terraform init -migrate-state -force-copy

In [ ]:
gcloud storage ls -r "gs://$STATE_BUCKET" --project "$PROJECT"

In [ ]:
gcloud storage cat "$STATE_OBJECT" --project "$PROJECT" \
  | grep -o '"result": "[^"]*"'

## Step 4 · Plan says no changes

In [ ]:
terraform plan

In [ ]:
ls -l terraform.tfstate*
grep -o '"result": "[^"]*"' terraform.tfstate.backup

In [ ]:
rm -f -- "${WORKDIR:?}/terraform.tfstate" "${WORKDIR:?}/terraform.tfstate.backup"

## Step 5 · Two terminals, one lock

In [ ]:
cp main.tf.sleep main.tf
grep -n -A6 'resource "time_sleep"' main.tf

In [ ]:
# First terminal
terraform apply -auto-approve -no-color > apply.log 2>&1 &
echo $! > apply.pid; echo "apply started"

In [ ]:
# Second terminal
sleep 20; tail -3 apply.log
terraform plan

In [ ]:
while kill -0 "$(cat apply.pid)" 2>/dev/null; do sleep 5; done
tail -4 apply.log

## Step 6 · The crashed run, and force-unlock

In [ ]:
terraform apply -auto-approve -no-color -replace=time_sleep.slow_apply > apply.log 2>&1 &
echo $! > apply.pid; echo "apply started"

In [ ]:
sleep 20; tail -3 apply.log
kill -9 "$(cat apply.pid)"; pkill -9 -f terraform-provider-time; echo "killed"

In [ ]:
terraform plan -no-color 2>&1 | tee plan-lock.log

In [ ]:
LOCK_ID=$(grep -Eo 'ID: *[0-9a-f-]+' plan-lock.log | head -1 | awk '{print $2}')
terraform force-unlock -force "${LOCK_ID:?}"

In [ ]:
terraform plan

## Step 7 · Versioning is the undo

In [ ]:
GOOD_GEN=$(gcloud storage objects describe "$STATE_OBJECT" --project "$PROJECT" --format='value(generation)')
echo "generation before the delete: $GOOD_GEN"
gcloud storage rm "gs://${STATE_BUCKET:?}/envs/dev/default.tfstate" \
  --project "${PROJECT:?}"

In [ ]:
terraform plan

In [ ]:
gcloud storage ls --all-versions --long \
  "gs://$STATE_BUCKET/envs/dev/" --project "$PROJECT"

In [ ]:
gcloud storage cp \
  "$STATE_OBJECT#${GOOD_GEN:?}" \
  "$STATE_OBJECT" \
  --project "$PROJECT"

In [ ]:
terraform plan

## Step 8 · Import what somebody built by hand

In [ ]:
cp legacy.tf.guess legacy.tf
cat legacy.tf

In [ ]:
terraform import google_storage_bucket.legacy \
  "$PROJECT/$LEGACY_BUCKET"

In [ ]:
terraform plan

In [ ]:
cp legacy.tf.matched legacy.tf
terraform plan

In [ ]:
cp annex.tf.staged annex.tf
cat annex.tf
terraform plan

In [ ]:
terraform apply -auto-approve

## Step 9 · Drift, and which side should win

In [ ]:
# The same label edit the runbook makes in the Console
gcloud storage buckets update "gs://$RAW_BUCKET" --project "$PROJECT" \
  --update-labels=owner=someone-at-2am

In [ ]:
terraform plan

In [ ]:
terraform apply -refresh-only -auto-approve

In [ ]:
terraform plan

In [ ]:
gcloud pubsub topics delete "${TOPIC:?}" --project "${PROJECT:?}" --quiet
terraform plan

In [ ]:
terraform apply -auto-approve

## Step 10 · Surgery on the record

In [ ]:
terraform state list
terraform state rm google_storage_bucket.legacy

In [ ]:
terraform plan

In [ ]:
rm -f -- "${WORKDIR:?}/legacy.tf"
terraform plan

In [ ]:
cp main.tf.renamed main.tf
terraform plan

In [ ]:
terraform state mv google_storage_bucket.raw google_storage_bucket.landing

In [ ]:
terraform plan

## Step 11 · Refuse to destroy, then permit it

In [ ]:
echo "quarterly totals, and nobody has a copy" > report.csv
gcloud storage cp report.csv "gs://$RAW_BUCKET/report.csv" \
  --project "$PROJECT"

In [ ]:
cp main.tf.protected main.tf
cd "${WORKDIR:?}" && terraform destroy -auto-approve

In [ ]:
cp main.tf.renamed main.tf
cd "${WORKDIR:?}" && terraform destroy -auto-approve -target=google_storage_bucket.landing

In [ ]:
cp main.tf.forced main.tf
terraform apply -auto-approve

In [ ]:
cd "${WORKDIR:?}" && terraform destroy -auto-approve

## After class

In [ ]:
gcloud storage rm --recursive --all-versions "gs://${STATE_BUCKET:?}" \
  --project "${PROJECT:?}"

gcloud storage rm --recursive "gs://${LEGACY_BUCKET:?}" \
  --project "${PROJECT:?}"

In [ ]:
gcloud storage rm --recursive "gs://${ANNEX_BUCKET:?}" \
  --project "${PROJECT:?}" 2>/dev/null || echo "annex bucket already removed by terraform destroy"

In [ ]:
gcloud storage ls --project "$PROJECT"